In [3]:
import numpy as np
from scipy.linalg import lu
n = int(input("n = "))
t = int(input("t (eps = 10^-t) = "))
eps = 10 ** (-t)

# Generare matrice SPD: A = B B^T

np.random.seed(0)
B = np.random.rand(n, n)
A = B @ B.T

Ainit = A.copy()   # salvăm matricea inițială
b = np.random.rand(n)


# LU folosind biblioteca

P, L_lib, U_lib = lu(Ainit)
xlib = np.linalg.solve(Ainit, b)

print("Descompunere LU (biblioteca):")
print("L =", L_lib)
print("U =", U_lib)
print("Solutia xlib =", xlib)

# Descompunere LDL^T (Choleski)
d = np.zeros(n)

for p in range(n):
    # calcul dp
    suma = 0.0
    for k in range(p):
        suma += d[k] * (A[p][k] ** 2)

    d[p] = A[p][p] - suma

    if abs(d[p]) < eps:
        raise ValueError("Matricea nu este pozitiv definita.")

    # calcul elemente L sub diagonala
    for i in range(p + 1, n):
        suma = 0.0
        for k in range(p):
            suma += d[k] * A[i][k] * A[p][k]

        A[i][p] = (A[i][p] - suma) / d[p]


# det(A) = det(L) det(D) det(L^T)
# det(L)=1, det(L^T)=1
# => det(A)=prod(di)
detA = np.prod(d)
print("Determinant A =", detA)

# Rezolvare Ax=b cu LDL^T
# L z = b (substitutie directa, diag=1)
z = np.zeros(n)
for i in range(n):
    suma = 0.0
    for j in range(i):
        suma += A[i][j] * z[j]
    z[i] = b[i] - suma

# D y = z
y = np.zeros(n)
for i in range(n):
    if abs(d[i]) < eps:
        raise ValueError("Impartire imposibila.")
    y[i] = z[i] / d[i]

# L^T x = y (substitutie inversa, diag=1)
xChol = np.zeros(n)
for i in reversed(range(n)):
    suma = 0.0
    for j in range(i + 1, n):
        suma += A[j][i] * xChol[j]
    xChol[i] = y[i] - suma

print("Solutia xChol =", xChol)

# Calcul manual Ainit * xChol
Ax = np.zeros(n)

for i in range(n):
    # element diagonal
    Ax[i] += A[i][i] * xChol[i]

    # partea superioară
    for j in range(i + 1, n):
        aij = A[i][j]   # partea superioară conține A inițial
        Ax[i] += aij * xChol[j]
        Ax[j] += aij * xChol[i]   # folosim simetria

norm1 = np.linalg.norm(Ax - b, 2)
norm2 = np.linalg.norm(xChol - xlib, 2)

print("||Ainit xChol - b||2 =", norm1)
print("||xChol - xlib||2 =", norm2)

n = 151
t (eps = 10^-t) = 8
Descompunere LU (biblioteca):
L = [[ 1.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.8030136   1.          0.         ...  0.          0.
   0.        ]
 [ 0.70409395  0.39791952  1.         ...  0.          0.
   0.        ]
 ...
 [ 0.75963583  0.25538884  0.27106535 ...  1.          0.
   0.        ]
 [ 0.68980295  0.40151753  0.21429202 ...  0.37909031  1.
   0.        ]
 [ 0.7158999   0.36493087  0.3422753  ... -0.24915393 -0.00171196
   1.        ]]
U = [[50.81324119 40.80372369 35.77729568 ... 40.5605784  35.05112381
  38.59955862]
 [ 0.         19.12441301  7.60997729 ...  7.39908316  7.67878713
   4.88416161]
 [ 0.          0.         18.36011533 ...  7.09775294  3.93442628
   4.97679112]
 ...
 [ 0.          0.          0.         ...  0.55819699  0.56556558
   1.35101817]
 [ 0.          0.          0.         ...  0.          0.18750292
  -0.18750573]
 [ 0.          0.          0.         ...  0.          0.
   0.12206881]